In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

In [4]:
file_path = 'C:/Users/mannu/2D Protien Folding/RS126.data.txt'
with open(file_path, 'r') as f:
    lines = f.readlines()

sequences, structures = [], []
for i in range(0, len(lines) - 1, 2):
    seq, struct = lines[i].strip(), lines[i+1].strip()
    if len(seq) == len(struct) and len(seq) > 0:
        sequences.append(seq)
        structures.append(struct)
window_size = 13
pad_length = window_size // 2
X_data, Y_data = [], []
for seq, struct in zip(sequences[:50], structures[:50]):
    padded_seq = ("X" * pad_length) + seq + ("X" * pad_length)
    for j in range(len(seq)):
        X_data.append(padded_seq[j : j + window_size])
        Y_data.append(struct[j])


In [12]:
(X_data[0:10], Y_data[0:10])

(['XXXXXXAPAFSVS',
  'XXXXXAPAFSVSP',
  'XXXXAPAFSVSPA',
  'XXXAPAFSVSPAS',
  'XXAPAFSVSPASG',
  'XAPAFSVSPASGA',
  'APAFSVSPASGAS',
  'PAFSVSPASGASD',
  'AFSVSPASGASDG',
  'FSVSPASGASDGQ'],
 ['C', 'C', 'E', 'E', 'E', 'E', 'E', 'C', 'C', 'C'])

In [23]:
alphabet = "ACDEFGHIKLMNPQRSTVWYX"
char_to_index = {char: idx for idx, char in enumerate(alphabet)}
vocab_size = len(alphabet)
X_flat = torch.zeros(len(X_data), window_size * vocab_size)
for row_idx, window in enumerate(X_data):
    for char_idx, char in enumerate(window):
        if char in char_to_index:
            col_idx = (char_idx * vocab_size) + char_to_index[char]
            X_flat[row_idx, col_idx] = 1.0
X_cnn = X_flat.view(-1, window_size, vocab_size).transpose(1, 2)
shape_mapping = {'C': 0, 'E': 1, 'H': 2}
Y_ints = [shape_mapping[shape] for shape in Y_data]
Y_tensor = torch.tensor(Y_ints, dtype=torch.long)

dataset = TensorDataset(X_cnn, Y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Data successfully formatted for CNN!")
print(f"X_cnn shape: {X_cnn.shape}  <-- Must be [Batch, 21, 13]")
print(f"Y_tensor shape: {Y_tensor.shape}")



Data successfully formatted for CNN!
X_cnn shape: torch.Size([8289, 21, 13])  <-- Must be [Batch, 21, 13]
Y_tensor shape: torch.Size([8289])


In [38]:
class MiniFoldCNN(nn.Module):
    def __init__(self, output_size=3):
        super(MiniFoldCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=21, out_channels=128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(3328, output_size)


    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.fc1(x)
        return x

model = MiniFoldCNN(output_size=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("CNN Model successfully initialized!")

CNN Model successfully initialized!


In [42]:
NUM_EPOCHS = 100

print("--- STARTING CNN TRAINING ---")
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    
    for batch_X, batch_Y in train_loader:
        # batch_X is shape [64, 21, 13]
        
        # 1. Forward Pass
        outputs = model(batch_X)
        
        # 2. Calculate Loss
        loss = criterion(outputs, batch_Y)
        
        # 3. Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    if(epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Loss: {avg_loss:.4f}")

print("--- TRAINING COMPLETE ---")

--- STARTING CNN TRAINING ---
Epoch [1/100] | Average Loss: 0.0024
Epoch [10/100] | Average Loss: 0.0005
Epoch [20/100] | Average Loss: 0.0002
Epoch [30/100] | Average Loss: 0.0001
Epoch [40/100] | Average Loss: 0.0001
Epoch [50/100] | Average Loss: 0.0000
Epoch [60/100] | Average Loss: 0.0000
Epoch [70/100] | Average Loss: 0.0000
Epoch [80/100] | Average Loss: 0.0000
Epoch [90/100] | Average Loss: 0.0000
Epoch [100/100] | Average Loss: 0.0000
--- TRAINING COMPLETE ---


In [44]:
def predict_protein_structure_cnn(protein_seq, trained_model, window_size=13, alphabet="ACDEFGHIKLMNPQRSTVWYX"):
    trained_model.eval()

    # -------------------------------------------------------------
    # STEP 2: PAD THE SEQUENCE WITH 'X's
    # -------------------------------------------------------------
    pad_length = window_size // 2 # 6 characters
    padded_seq = ("X" * pad_length) + protein_seq + ("X" * pad_length)
    char_to_idx = {char: i for i, char in enumerate(alphabet)}
    vocab_size = len(alphabet) # 21
    protein_len = len(protein_seq)

    # -------------------------------------------------------------
    # STEP 3: SLIDING WINDOW & ONE-HOT ENCODING
    # -------------------------------------------------------------
    # Build a flat matrix first: [Protein_Length, 13 * 21] -> e.g. [30, 273]
    X_flat = torch.zeros(protein_len, window_size * vocab_size)
    
    for i in range(protein_len):
        window = padded_seq[i : i + window_size]
        for char_idx, char in enumerate(window):
            if char in char_to_idx:
                col_idx = (char_idx * vocab_size) + char_to_idx[char]
                X_flat[i, col_idx] = 1.0

    # -------------------------------------------------------------
    # STEP 4: RESHAPE FOR CNN -> [Batch, Channels, Length]
    # [30, 273] -> [30, 13, 21] -> Transpose to [30, 21, 13]
    # -------------------------------------------------------------
    X_inference = X_flat.view(-1, window_size, vocab_size).transpose(1, 2)

    # -------------------------------------------------------------
    # STEP 5: FORWARD PASS (No Gradients needed)
    # -------------------------------------------------------------
    with torch.no_grad():
        # Pass all 30 windows through the CNN simultaneously!
        outputs = trained_model(X_inference) # Output shape: [30, 3]
        
        # Find which class (0, 1, or 2) had the highest score for each amino acid
        predicted_indices = torch.argmax(outputs, dim=1) # Shape: [30]

    # -------------------------------------------------------------
    # STEP 6: MAP INTEGERS BACK TO BIOLOGICAL LETTERS
    # -------------------------------------------------------------
    int_to_shape = {0: 'C', 1: 'E', 2: 'H'}
    predicted_chars = [int_to_shape[int(idx.item())] for idx in predicted_indices]

    return "".join(predicted_chars)


# =============================================================
# RUN THE INFERENCE TEST & CALCULATE ACCURACY
# =============================================================
test_input      = "FVNQHLCGSHLVEALYLVCGERGFFYTPKA"
expected_output = "CCCCCCCCHHHHHHHHHHHHHHCECCCCCC"

# Get CNN Prediction
prediction = predict_protein_structure_cnn(test_input, model)

# Calculate accuracy score
matches = sum(1 for p, e in zip(prediction, expected_output) if p == e)
accuracy = (matches / len(expected_output)) * 100

print("--- CNN INFERENCE TEST RESULTS ---")
print(f"Input Sequence:   {test_input}")
print(f"Expected Ground:  {expected_output}")
print(f"CNN Prediction:   {prediction}")
print(f"Accuracy:         {accuracy:.1f}% ({matches}/{len(expected_output)} correct amino acids)")

--- CNN INFERENCE TEST RESULTS ---
Input Sequence:   FVNQHLCGSHLVEALYLVCGERGFFYTPKA
Expected Ground:  CCCCCCCCHHHHHHHHHHHHHHCECCCCCC
CNN Prediction:   CCEEEECHHHHHHEEEEEECCCCCCCCCCC
Accuracy:         50.0% (15/30 correct amino acids)
